# Exploratory Data Analysis — Crypto Volatility

This notebook loads the feature Parquet, explores distributions,
and uses percentile analysis to choose a volatility spike threshold.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.float_format", "{:.6f}".format)

## 1. Load Features

In [ ]:
df = pd.read_parquet("../data/processed/features.parquet")
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df):,}")
print(f"Pairs: {df['product_id'].unique()}")
print(f"Time range: {df['timestamp'].min()} → {df['timestamp'].max()}")
df.head()

## 2. Summary Statistics

In [ ]:
feature_cols = [
    "midprice", "midprice_log_return", "bid_ask_spread",
    "spread_bps", "rolling_volatility", "trade_intensity",
    "order_book_imbalance",
]
df[feature_cols].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99])

## 3. Feature Distributions

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(14, 10))
axes = axes.ravel()

for i, col in enumerate(feature_cols):
    ax = axes[i]
    df[col].hist(bins=80, ax=ax, edgecolor="none", alpha=0.7)
    ax.set_title(col)
    ax.axvline(df[col].median(), color="red", ls="--", lw=1, label="median")
    ax.legend(fontsize=8)

for j in range(len(feature_cols), len(axes)):
    axes[j].set_visible(False)

plt.tight_layout()
plt.show()

## 4. Time Series Plots

In [ ]:
for pid in df["product_id"].unique():
    sub = df[df["product_id"] == pid]
    fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

    axes[0].plot(sub["timestamp"], sub["midprice"], lw=0.5)
    axes[0].set_ylabel("Midprice")
    axes[0].set_title(f"{pid} — Time Series")

    axes[1].plot(sub["timestamp"], sub["rolling_volatility"], lw=0.5, color="orange")
    axes[1].set_ylabel("Rolling Volatility")

    axes[2].plot(sub["timestamp"], sub["bid_ask_spread"], lw=0.5, color="green")
    axes[2].set_ylabel("Bid-Ask Spread")
    axes[2].set_xlabel("Time")

    plt.tight_layout()
    plt.show()

## 5. Future Volatility & Spike Threshold

Compute the forward-looking volatility (rolling std of midprice log-returns
over the next 60 seconds) and use percentile plots to choose a threshold.

In [ ]:
HORIZON = 60  # seconds

def compute_future_volatility(group):
    """For each row, compute std of log-returns over the next HORIZON seconds."""
    ts = group["timestamp"].values
    returns = group["midprice_log_return"].values
    sigma_future = np.full(len(group), np.nan)

    for i in range(len(group)):
        t0 = ts[i]
        horizon_end = t0 + np.timedelta64(HORIZON, "s")
        mask = (ts > t0) & (ts <= horizon_end)
        future_rets = returns[mask]
        if len(future_rets) >= 2:
            sigma_future[i] = np.std(future_rets, ddof=1)

    group["sigma_future"] = sigma_future
    return group

df = df.groupby("product_id", group_keys=False).apply(compute_future_volatility)
df_labeled = df.dropna(subset=["sigma_future"]).copy()
print(f"Rows with valid sigma_future: {len(df_labeled):,} / {len(df):,}")

In [ ]:
percentiles = np.arange(50, 100, 1)
pct_values = np.percentile(df_labeled["sigma_future"], percentiles)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(percentiles, pct_values, marker="o", ms=3)
ax.set_xlabel("Percentile")
ax.set_ylabel("sigma_future")
ax.set_title("Percentile Plot of Future Volatility (60s horizon)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

for p in [75, 80, 85, 90, 95, 99]:
    v = np.percentile(df_labeled["sigma_future"], p)
    n = (df_labeled["sigma_future"] >= v).sum()
    print(f"  P{p:2d} = {v:.8f}  → {n:,} spikes ({n/len(df_labeled)*100:.1f}%)")

## 6. Choose Threshold & Create Labels

Pick a threshold based on the percentile plot above.
A good starting point is around P85-P90 to get a ~10-15% positive rate.

In [ ]:
THRESHOLD = np.percentile(df_labeled["sigma_future"], 90)  # adjust after reviewing plots
print(f"Chosen threshold (P90): {THRESHOLD:.8f}")

df_labeled["label"] = (df_labeled["sigma_future"] >= THRESHOLD).astype(int)

print(f"\nLabel distribution:")
print(df_labeled["label"].value_counts())
print(f"\nPositive rate: {df_labeled['label'].mean():.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
df_labeled["sigma_future"].hist(bins=100, ax=ax, edgecolor="none", alpha=0.7)
ax.axvline(THRESHOLD, color="red", ls="--", lw=2, label=f"threshold = {THRESHOLD:.6f}")
ax.set_xlabel("sigma_future")
ax.set_ylabel("Count")
ax.set_title("Distribution of Future Volatility with Chosen Threshold")
ax.legend()
plt.tight_layout()
plt.show()

## 7. Correlation Matrix

In [ ]:
corr_cols = feature_cols + ["sigma_future", "label"]
corr = df_labeled[corr_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr_cols)))
ax.set_yticks(range(len(corr_cols)))
ax.set_xticklabels(corr_cols, rotation=45, ha="right", fontsize=9)
ax.set_yticklabels(corr_cols, fontsize=9)
fig.colorbar(im)
ax.set_title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

## 8. Save Labeled Dataset

In [ ]:
df_labeled.to_parquet("../data/processed/features.parquet", index=False)
print(f"Saved {len(df_labeled):,} rows with labels to features.parquet")